# 03 — Recovery analysis

Per-drift-event recovery metrics (RT, AUD, min-F1, no-recovery) across phases and seeds.
Run the experiments first (`make run-all`), then execute this notebook.

Analysis only — pipelines live in `src/`, never here.

In [ ]:
from pathlib import Path

import pandas as pd
from scripts.make_report import load_recovery_frames

from driftgan.config import RESULTS_DIR
from driftgan.metrics.recovery import aggregate_recovery

recovery = load_recovery_frames(Path(RESULTS_DIR))
{phase: len(df) for phase, df in recovery.items()}

In [ ]:
# Aggregated table: mean / median / IQR of RT and AUD per phase
pd.concat({phase: aggregate_recovery(df) for phase, df in recovery.items()}, names=["phase"])

In [ ]:
from driftgan.viz.plots import recovery_boxplots, rt_cdf

recovery_boxplots(recovery, "RT");

In [ ]:
recovery_boxplots(recovery, "AUD");

In [ ]:
rt_cdf(recovery);

In [ ]:
# Paired Wilcoxon between phases (valid only under the shared drift schedule)
from itertools import combinations

from driftgan.experiments.stats import paired_wilcoxon

rows = []
for pa, pb in combinations(sorted(recovery), 2):
    for metric in ("RT", "AUD"):
        rows.append(paired_wilcoxon(recovery[pa], recovery[pb], metric, pa, pb).__dict__)
pd.DataFrame(rows)

## Interpretation checklist

- Is `no_recovery_rate` comparable across phases before comparing RT?
- Are `excluded_windows_pct` values similar? Large asymmetries mean the F1 window saw different label mixes.
- Effect sizes (Cliff's delta, r) matter more than p-values with many events.